In [ ]:
results = """
================================================================================================================================================================
MPC + SVGP EXPERIMENT RESULTS
================================================================================================================================================================
Track                   mu   Seed     Ts   Steps Collision   Laps  Time(s)   AvgSpd     RMSE_w     RMSE_b  Wall(s)   ms/act Status       Lap times (s)
----------------------------------------------------------------------------------------------------------------------------------------------------------------
Sepang_10489        1.0000     42  0.020    5189 No             2   103.78     4.69   0.001041   0.021785    713.2 77.3328ms OK           [52.2s  51.6s]
Sepang_10489        0.9000     42  0.020    5190 No             2   103.80     4.69   0.001041   0.021783    707.9 76.9606ms OK           [52.2s  51.6s]
Sepang_10489        0.8000     42  0.020    5197 No             2   103.94     4.69   0.001040   0.021768    713.9 77.6886ms OK           [52.3s  51.7s]
Sepang_10489        0.7000     42  0.020    5203 No             2   104.06     4.68   0.001039   0.021756    718.6 78.0015ms OK           [52.3s  51.7s]
Sepang_10489        0.6000     42  0.020    1055 YES            0    21.10    23.08   3.736436   0.048314    241.2 77.6904ms OK           []
Sepang_10489        0.5000     42  0.020     545 YES            0    10.90    44.68   6.847909   0.067221    184.9 81.7734ms OK           []
Sepang_10489        0.4000     42  0.020     557 YES            0    11.14    43.71   5.073545   0.066493    187.1 81.8650ms OK           []
Sepang_10489        0.3000     42  0.020     515 YES            0    10.30    47.28   1.975052   0.069151    180.9 80.4492ms OK           []
Sepang_10489        0.2000     42  0.020     336 YES            0     6.72    72.47   3.663312   0.085611    158.6 84.2360ms OK           []
Sepang_10489        0.1000     42  0.020     305 YES            0     6.10    79.83   2.014875   0.089857    155.3 84.1512ms OK           []
YasMarina_10489     1.0000     42  0.020    4648 No             2    92.96     4.28   0.017304   0.023371    595.2 73.6312ms OK           [46.8s  46.2s]
YasMarina_10489     0.9000     42  0.020    4655 No             2    93.10     4.28   0.017291   0.023354    596.9 73.7818ms OK           [46.8s  46.3s]
YasMarina_10489     0.8000     42  0.020    4666 No             2    93.32     4.27   0.017271   0.023326    598.8 74.0920ms OK           [46.9s  46.4s]
YasMarina_10489     0.7000     42  0.020    4669 No             2    93.38     4.26   0.017265   0.023319    601.2 74.6768ms OK           [47.0s  46.4s]
YasMarina_10489     0.6000     42  0.020     365 YES            0     7.30    54.52   4.583740   0.083401    152.8 81.7332ms OK           []
YasMarina_10489     0.5000     42  0.020     304 YES            0     6.08    65.47   7.898753   0.091386    146.3 82.2611ms OK           []
YasMarina_10489     0.4000     42  0.020     305 YES            0     6.10    65.25   7.110761   0.091236    149.6 85.3654ms OK           []
YasMarina_10489     0.3000     42  0.020     200 YES            0     4.00    99.51  10.002029   0.112668    131.9 88.4400ms OK           []
YasMarina_10489     0.2000     42  0.020     204 YES            0     4.08    97.56   7.336121   0.111558    138.1 97.3928ms OK           []
YasMarina_10489     0.1000     42  0.020     169 YES            0     3.38   117.76   1.525564   0.122566    131.1 94.9277ms OK           []
================================================================================================================================================================
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

def parse_results(text):
    rows = []
    for line in text.strip().split("\n"):
        parts = line.split()
        if len(parts) < 6:
            continue
        if "Sepang" not in parts[0] and "YasMarina" not in parts[0]:
            continue
        track = parts[0].split("_")[0]
        collision = 1 if parts[5] == "YES" else 0
        lap_times = []
        match = re.search(r'\[([^\]]+)\]', line)
        if match and match.group(1).strip():
            lap_times = [float(t.replace('s', '')) for t in match.group(1).split()]
        # Extract ms/act (parts[12])
        ms_act = float(parts[12].replace('ms', '')) if len(parts) > 12 else np.nan
        rows.append({
            "track": track,
            "mu": float(parts[1]),
            "seed": int(parts[2]),
            "collision": collision,
            "ms_per_act": ms_act,
            "lap_times": lap_times,
            "fastest_lap": min(lap_times) if lap_times else np.nan,
            "avg_lap": np.mean(lap_times) if lap_times else np.nan,
        })
    return rows

df = pd.DataFrame(parse_results(results))
print(f"Parsed {len(df)} rows ({df['collision'].sum()} collisions)")
print(f"Tracks: {sorted(df['track'].unique())}")
print(f"Mus: {sorted(df['mu'].unique())}")
print(f"Seeds: {sorted(df['seed'].unique())}")

print(f"\n{'='*50}")
print(f"ms/act statistics:")
print(f"{'='*50}")
print(f"  Overall mean:   {df['ms_per_act'].mean():.2f} ms")
print(f"  Overall median: {df['ms_per_act'].median():.2f} ms")
print(f"  Overall std:    {df['ms_per_act'].std():.2f} ms")
print(f"  Min:            {df['ms_per_act'].min():.2f} ms")
print(f"  Max:            {df['ms_per_act'].max():.2f} ms")

for track in sorted(df["track"].unique()):
    sub = df[df["track"] == track]
    print(f"\n  {track}:  mean={sub['ms_per_act'].mean():.2f} ms  median={sub['ms_per_act'].median():.2f} ms")


In [ ]:
# --- Crash percentage bar chart: one subplot per track ---
crash_pct = df.groupby(["track", "mu"])["collision"].mean() * 100
crash_pct = crash_pct.reset_index().rename(columns={"collision": "crash_pct"})

tracks = sorted(df["track"].unique())
mus = sorted(df["mu"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, track in zip(axes, tracks):
    sub = crash_pct[crash_pct["track"] == track].set_index("mu").reindex(mus).fillna(0)
    colors = plt.cm.RdYlGn_r(sub["crash_pct"].values / 100)
    bars = ax.bar([f"{m:.1f}" for m in mus], sub["crash_pct"].values, color=colors, edgecolor="gray", linewidth=0.5)

    for bar, val in zip(bars, sub["crash_pct"].values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                    f"{val:.0f}%", ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.set_xlabel("Friction coefficient (μ)", fontsize=12)
    ax.set_title(f"{track}", fontsize=13, fontweight="bold")
    ax.set_ylim(0, 110)

axes[0].set_ylabel("Crash %", fontsize=12)
fig.suptitle("MPC (Jetson Orin) — Crash Percentage", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# --- Fastest lap time bar chart: one subplot per track ---
df_ok = df[df["collision"] == 0]

tracks = sorted(df["track"].unique())
mus = sorted(df["mu"].unique())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, track in zip(axes, tracks):
    sub = df_ok[df_ok["track"] == track].groupby("mu")["fastest_lap"].mean()
    sub = sub.reindex(mus)

    vals = sub.values.copy()
    valid = ~np.isnan(vals)
    x_labels = [f"{m:.1f}" for m in mus]

    if valid.any():
        vmin, vmax = np.nanmin(vals), np.nanmax(vals)
        norm_vals = np.where(valid, (vals - vmin) / (vmax - vmin + 1e-8), 0)
        colors = [plt.cm.viridis_r(n) if v else (0.9, 0.9, 0.9, 1.0) for n, v in zip(norm_vals, valid)]
    else:
        colors = [(0.9, 0.9, 0.9, 1.0)] * len(mus)

    bar_vals = np.where(valid, vals, 0)
    bars = ax.bar(x_labels, bar_vals, color=colors, edgecolor="gray", linewidth=0.5)

    for bar, val, v in zip(bars, vals, valid):
        if v:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f"{val:.1f}", ha="center", va="bottom", fontsize=8, fontweight="bold")
        else:
            ax.text(bar.get_x() + bar.get_width()/2, ax.get_ylim()[1] * 0.05,
                    "X", ha="center", va="bottom", fontsize=10, fontweight="bold", color="red")

    ax.set_xlabel("Friction coefficient (μ)", fontsize=12)
    ax.set_ylabel("Fastest lap time (s)", fontsize=12)
    ax.set_title(f"{track}", fontsize=13, fontweight="bold")

fig.suptitle("MPC (Jetson Orin) — Fastest Lap Time (non-crashed trials)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# --- Collision details ---
crashed = df[df["collision"] == 1]

if len(crashed) == 0:
    print("No collisions in any configuration!")
else:
    for track in sorted(crashed["track"].unique()):
        sub = crashed[crashed["track"] == track].sort_values(["mu", "seed"])
        grouped = sub.groupby("mu").agg(
            crashes=("collision", "sum"),
            seeds=("seed", lambda s: sorted(s.tolist())),
        ).reset_index()

        print(f"\n{'='*60}")
        print(f"  {track} — mu values with collisions")
        print(f"{'='*60}")
        print(f"  {'mu':>6s}  {'crashes':>7s}   seeds")
        print(f"  {'-'*40}")
        for _, row in grouped.iterrows():
            seeds_str = ", ".join(str(s) for s in row["seeds"])
            print(f"  {row['mu']:6.2f}  {row['crashes']:>7d}     [{seeds_str}]")


In [ ]:
# --- LaTeX table rows and crash % printout for filling in paper table ---
mus = sorted(df["mu"].unique(), reverse=True)
tracks = sorted(df["track"].unique())
df_ok = df[df["collision"] == 0]

print("MPC (Jetson Orin) Crash Percentage per mu")
print("=" * 80)

for track in tracks:
    crash = df[df["track"] == track].groupby("mu")["collision"].mean() * 100
    avg = crash.mean()
    print(f"{track:12s}  " + "  ".join([f"μ={m:.1f}: {crash.get(m,0):3.0f}%" for m in mus]) + f"  AVG: {avg:.0f}%")

print()
print("Combined (both maps):")
crash_combined = df.groupby("mu")["collision"].mean() * 100
avg_combined = crash_combined.mean()
print("  " + "  ".join([f"μ={m:.1f}: {crash_combined.get(m,0):3.0f}%" for m in mus]) + f"  AVG: {avg_combined:.0f}%")

print()
print("--- For LaTeX table (copy-paste ready, combined) ---")
row_vals = " & ".join([f"{crash_combined.get(m, 0):.0f}\\%" for m in mus])
print(f"MPC & {row_vals} & {avg_combined:.0f}\\% \\\\")

print()
print("--- Per track rows ---")
for track in tracks:
    crash = df[df["track"] == track].groupby("mu")["collision"].mean() * 100
    avg = crash.mean()
    row_vals = " & ".join([f"{crash.get(m, 0):.0f}\\%" for m in mus])
    print(f"MPC ({track}) & {row_vals} & {avg:.0f}\\% \\\\")

print()
print("=" * 80)
print("MPC (Jetson Orin) Fastest Lap per mu")
print("=" * 80)
fastest = df_ok.groupby(["track", "mu"])["fastest_lap"].mean()

for track in tracks:
    vals_str = "  ".join([f"μ={m:.1f}: {fastest.get((track, m), np.nan):5.1f}s" if not np.isnan(fastest.get((track, m), np.nan)) else f"μ={m:.1f}:    --" for m in mus])
    print(f"{track:12s}  {vals_str}")

# LaTeX booktabs tables
print()
print(r"% Requires: \usepackage{booktabs}")
col_spec = "c" + "c" * len(mus)
header = r"Track & " + " & ".join([f"{m:.1f}" for m in mus]) + r" \\"

print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{MPC (Jetson Orin) crash percentage (\%) by friction coefficient.}")
print(r"\label{tab:mpc_jetson_crash_pct}")
print(f"\\begin{{tabular}}{{{col_spec}}}")
print(r"\toprule")
print(header)
print(r"\midrule")
for track in tracks:
    crash = df[df["track"] == track].groupby("mu")["collision"].mean() * 100
    vals = " & ".join([f"{crash.get(m, 0):.0f}" if crash.get(m, 0) > 0 else "0" for m in mus])
    print(f"{track} & {vals}" + r" \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")

print()
print(r"\begin{table}[ht]")
print(r"\centering")
print(r"\caption{MPC (Jetson Orin) fastest lap time (s) by friction coefficient.}")
print(r"\label{tab:mpc_jetson_fastest_lap}")
print(f"\\begin{{tabular}}{{{col_spec}}}")
print(r"\toprule")
print(header)
print(r"\midrule")
for track in tracks:
    vals = " & ".join([f"{fastest.get((track, m), np.nan):.1f}" if not np.isnan(fastest.get((track, m), np.nan)) else "--" for m in mus])
    print(f"{track} & {vals}" + r" \\")
print(r"\bottomrule")
print(r"\end{tabular}")
print(r"\end{table}")
